In [79]:
!pip install ucimlrepo

In [80]:
import pandas as pd
import numpy as np
import joblib

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [81]:
dataset = fetch_ucirepo(id=222)

X = dataset.data.features
y = dataset.data.targets

df = pd.concat([X, y], axis=1)

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


In [82]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer



def preprocess_data(df, target_column):

    X = df.drop(columns=[target_column])
    y = df[target_column]

    categorical_cols = X.select_dtypes(include=["object", "category"]).columns
    numeric_cols = X.select_dtypes(include=["number"]).columns

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ])

    X = preprocessor.fit_transform(X)

    target_encoder = None

    if y.dtype == "object" or str(y.dtype) == "category":
        target_encoder = LabelEncoder()
        y = target_encoder.fit_transform(y)

    return X, y, preprocessor, target_encoder

In [83]:
def train_models(X_train, y_train):

    models = {

        "Logistic Regression":
            LogisticRegression(max_iter=1000),

        "Decision Tree":
            DecisionTreeClassifier(random_state=42),

        "KNN":
            KNeighborsClassifier(),

        "Naive Bayes":
            GaussianNB(),

        "Random Forest":
            RandomForestClassifier(
                n_estimators=100,
                random_state=42
            )

    }

    for model in models.values():

        model.fit(X_train, y_train)

    return models

In [84]:
def evaluate_models(models, X_test, y_test):

    results = []

    reports = {}

    for name, model in models.items():

        y_pred = model.predict(X_test)

        row = {}

        row["Model"] = name
        row["Accuracy"] = accuracy_score(y_test, y_pred)
        row["Precision"] = precision_score(y_test, y_pred)
        row["Recall"] = recall_score(y_test, y_pred)
        row["F1"] = f1_score(y_test, y_pred)
        row["MCC"] = matthews_corrcoef(y_test, y_pred)

        if hasattr(model, "predict_proba"):

            y_prob = model.predict_proba(X_test)[:,1]

            row["AUC"] = roc_auc_score(y_test, y_prob)

        else:

            row["AUC"] = None

        results.append(row)

        reports[name] = {

            "confusion_matrix":
                confusion_matrix(y_test, y_pred),

            "classification_report":
                classification_report(
                    y_test,
                    y_pred,
                    output_dict=True
                )

        }

    results_df = pd.DataFrame(results)

    return results_df, reports

In [85]:
def run_pipeline(df, target_column):

    X, y, preprocessor, target_encoder = preprocess_data(
    df,
    target_column)

    X_train, X_test, y_train, y_test = train_test_split(

        X,
        y,

        test_size=0.20,

        random_state=42,

        stratify=y

    )

    models = train_models(
        X_train,
        y_train
    )

    results_df, reports = evaluate_models(
        models,
        X_test,
        y_test
    )

    return (
        models,
        results_df,
        reports,
        X_test,
        y_test,
        scaler,
        encoders,
        target_encoder
    )

In [ ]:
models, results_df, reports, X_test, y_test, scaler, encoders, target_encoder = run_pipeline(
    df,
    "y"
)

In [ ]:
results_df

In [ ]:
reports["Random Forest"]["Confusion Matrix"]

In [ ]:
pd.DataFrame(
    reports["Random Forest"]["Classification Report"]
).transpose()

In [ ]:
os.makedirs("saved_models", exist_ok=True)

for name, model in models.items():

    filename = name.lower().replace(" ", "_") + ".pkl"

    joblib.dump(
        model,
        f"saved_models/{filename}"
    )

joblib.dump(preprocessor, "saved_models/preprocessor.pkl")

joblib.dump(
    target_encoder,
    "saved_models/target_encoder.pkl"
)

results_df.to_csv(
    "model_results.csv",
    index=False
)

print("Everything saved successfully!")

In [ ]:
test_df = pd.DataFrame(X_test)

test_df["Target"] = y_test

test_df.to_csv(
    "test_data.csv",
    index=False
)

print("Test data saved.")

In [ ]:
test_df = pd.DataFrame(X_test)

test_df["Target"] = y_test

test_df.to_csv(
    "test_data.csv",
    index=False
)

print("Test data saved.")